## Benchmark 3: Helmholtz PDE

In [ ]:
from jaxkan.models.KAN import KAN
from jaxkan.pikan.pde import get_helmholtz_res
from jaxkan.grids import (
    AdaptationState, 
    UniformDensity, 
    CurvatureDensity,
    MixedAdaptation, 
    ScheduledTrigger,
    reset_after_adaptation,
    update_state
)

import jax
import jax.numpy as jnp
from flax import nnx
import optax

import numpy as np
import pandas as pd
import time
from datetime import datetime


In [ ]:
@nnx.jit
def compute_curvature(model, X):
    
    epsilon=1e-3
    n_in = X.shape[1]
    curvatures = jnp.zeros(X.shape[0])
    
    for dim in range(n_in):
        h = jnp.zeros((1, n_in))
        h = h.at[0, dim].set(epsilon)
        
        X_plus = X + h
        X_minus = X - h
        
        f_center = model(X)
        f_plus = model(X_plus)
        f_minus = model(X_minus)
        
        second_deriv = (f_plus - 2 * f_center + f_minus) / (epsilon ** 2)
        curvatures = curvatures + jnp.sum(jnp.abs(second_deriv), axis=1)
    
    return curvatures

## Helper Functions

In [ ]:
def generate_helmholtz_reference(a1, a2, n_eval=100):
    
    x = jnp.linspace(-1, 1, n_eval)
    y = jnp.linspace(-1, 1, n_eval)
    X, Y = jnp.meshgrid(x, y, indexing='ij')
    coords = jnp.stack([X.flatten(), Y.flatten()], axis=1)
    
    # Analytical solution
    refsol = jnp.sin(jnp.pi * a1 * X) * jnp.sin(jnp.pi * a2 * Y)
    
    return refsol.flatten(), coords


def get_collocs_helmholtz(N, a1, a2):
    
    # PDE collocation points
    x_pde = jnp.linspace(-1, 1, N)
    y_pde = jnp.linspace(-1, 1, N)
    X_pde, Y_pde = jnp.meshgrid(x_pde, y_pde, indexing='ij')
    pde_collocs = jnp.stack([X_pde.flatten(), Y_pde.flatten()], axis=1) # (N^2, 2)

    # Boundary conditions: u(-1,y) = u(1,y) = u(x,-1) = u(x,1) = 0
    x_bc_1 = jnp.array([-1.0])
    X_bc_1, Y_bc_1 = jnp.meshgrid(x_bc_1, y_pde, indexing='ij')
    bc_1 = jnp.stack([X_bc_1.flatten(), Y_bc_1.flatten()], axis=1) # (N, 2)
    bc_1_data = jnp.zeros(bc_1.shape[0]).reshape(-1,1) # (N, 1)

    x_bc_2 = jnp.array([1.0])
    X_bc_2, Y_bc_2 = jnp.meshgrid(x_bc_2, y_pde, indexing='ij')
    bc_2 = jnp.stack([X_bc_2.flatten(), Y_bc_2.flatten()], axis=1) # (N, 2)
    bc_2_data = jnp.zeros(bc_2.shape[0]).reshape(-1,1) # (N, 1)

    y_bc_3 = jnp.array([-1.0])
    X_bc_3, Y_bc_3 = jnp.meshgrid(x_pde, y_bc_3, indexing='ij')
    bc_3 = jnp.stack([X_bc_3.flatten(), Y_bc_3.flatten()], axis=1) # (N, 2)
    bc_3_data = jnp.zeros(bc_3.shape[0]).reshape(-1,1) # (N, 1)

    y_bc_4 = jnp.array([1.0])
    X_bc_4, Y_bc_4 = jnp.meshgrid(x_pde, y_bc_4, indexing='ij')
    bc_4 = jnp.stack([X_bc_4.flatten(), Y_bc_4.flatten()], axis=1) # (N, 2)
    bc_4_data = jnp.zeros(bc_4.shape[0]).reshape(-1,1) # (N, 1)

    bc_collocs = jnp.concatenate([bc_1, bc_2, bc_3, bc_4], axis=0)
    bc_data = jnp.concatenate([bc_1_data, bc_2_data, bc_3_data, bc_4_data], axis=0)

    return pde_collocs, bc_collocs, bc_data


def compute_helmholtz_error(model, a1, a2):
    
    refsol, coords = generate_helmholtz_reference(a1, a2)
    
    # Model prediction
    u_pred = model(coords).flatten()
    
    # Compute relative L^2 error
    numerator = jnp.sqrt(jnp.mean((u_pred - refsol)**2))
    denominator = jnp.sqrt(jnp.mean(refsol**2))
    
    return float(numerator / denominator)


@nnx.jit(static_argnames=['pde_res_fn'])
def train_step(model, optimizer, pde_collocs, bc_collocs, bc_data, pde_res_fn):
    
    def loss_fn(model):
        # PDE residual loss
        pde_res = pde_res_fn(model, pde_collocs)
        total_loss = jnp.mean(pde_res**2)
        
        # BC loss
        bc_res = model(bc_collocs) - bc_data
        total_loss += jnp.mean(bc_res**2)
        
        return total_loss
    
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    
    return loss

## Training Function

In [ ]:
def run_training(
    a1,
    a2,
    k,
    pde_res_fn,
    pde_collocs,
    bc_collocs,
    bc_data,
    architecture,
    method,
    seed,
    num_epochs,
    learning_rate,
    grid_schedule,
    verbose=False
):
    
    start_time = time.time()
    
    # Create model
    req_params = {'k': 3, 'G': 3, 'init_scheme': {'type': 'glorot_fine'}}
    
    model = KAN(
        layer_dims=architecture,
        layer_type='spline',
        required_parameters=req_params,
        seed=seed
    )
    
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)
    
    # Setup adaptation framework
    state = AdaptationState()
    trigger = ScheduledTrigger(epochs=list(grid_schedule.keys()))
    
    # Configure IDF and strategy based on method
    if method == 'input_adaptive':
        idf = UniformDensity()
        strategy = MixedAdaptation(grid_e=0.0)
    elif method == 'curvature_adaptive':
        idf = CurvatureDensity()
        strategy = MixedAdaptation(grid_e=0.0)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Training loop
    for epoch in range(num_epochs):
        # Check for grid adaptation
        if trigger(state):
            G_new = grid_schedule[epoch]
            
            # For curvature-based, compute curvatures on PDE collocation points
            if method == 'curvature_adaptive':
                curvatures = compute_curvature(model, pde_collocs)
                model.update_grids(
                    pde_collocs,
                    idf=idf,
                    strategy=strategy,
                    grid_size_new=G_new,
                    curvatures=curvatures
                )
            else:
                model.update_grids(
                    pde_collocs,
                    idf=idf,
                    strategy=strategy,
                    grid_size_new=G_new
                )
            
            # Reset optimizer after grid change
            optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)
            state = reset_after_adaptation(state)
        
        # Training step
        loss = train_step(model, optimizer, pde_collocs, bc_collocs, bc_data, pde_res_fn)
        state = update_state(state, loss=loss)
    
    # Final evaluation
    rel_l2_error = compute_helmholtz_error(model, a1, a2)
    wall_time = time.time() - start_time
    
    if verbose:
        print(f"\n\t  Rel L^2 error: {rel_l2_error:.6e} (time: {wall_time:.1f}s)")
    
    return {
        'rel_l2_error': rel_l2_error,
        'wall_time': wall_time
    }

## Experiment Configuration

In [ ]:
# Frequency configurations to test
# Each tuple is (a1, a2)
alphas = [
    (1.0, 1.0),
    (1.0, 2.0),
    (2.0, 2.0),
    (2.0, 4.0)
]

# Fixed parameters
k = 1.0  # Wave number
colloc_N = 64  # N points per dimension
architecture = [2, 6, 6, 1]

# Methods to compare
METHODS = ['input_adaptive', 'curvature_adaptive']

# Number of random seeds
SEEDS = [3, 5, 9]
N_SEEDS = len(SEEDS)

# Training configuration
TRAINING_CONFIG = {
    'num_epochs': 5000,
    'learning_rate': 1e-3,
    'grid_schedule': {0: 3, 1000: 6, 2000: 9, 3000: 12}
}

# Print configuration summary
print("Helmholtz Benchmark Configuration:")
print(f"  Frequency configs: {alphas}")
print(f"  Methods: {METHODS}")
print(f"  Seeds: {N_SEEDS}")
print(f"  Architecture: {architecture}")
print(f"\nTraining Config:")
print(f"  Epochs: {TRAINING_CONFIG['num_epochs']}")
print(f"  Grid schedule: {TRAINING_CONFIG['grid_schedule']}")
print(f"\nTotal experiments: {len(alphas)} × {len(METHODS)} × {N_SEEDS}")
print(f"  = {len(alphas) * len(METHODS) * N_SEEDS} training runs")

## Run Benchmark

In [ ]:
# Storage for results
results = []

total_runs = len(alphas) * len(METHODS) * N_SEEDS
current_run = 0

print(f"Starting Helmholtz benchmark: {total_runs} total runs")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

for (a1, a2) in alphas:
    print(f"\nRunning Experiments for Helmholtz with a1={a1}, a2={a2}")
    
    # Get PDE residual function for this configuration
    pde_res_fn = get_helmholtz_res(a1=a1, a2=a2, k=k)
    
    # Generate collocation points
    pde_collocs, bc_collocs, bc_data = get_collocs_helmholtz(colloc_N, a1, a2)
    
    for method in METHODS:
        print(f"    Method: {method}", end=" ")
        
        method_errors = []
        
        for seed in SEEDS:
            current_run += 1
            
            # Run training
            result = run_training(
                a1=a1,
                a2=a2,
                k=k,
                pde_res_fn=pde_res_fn,
                pde_collocs=pde_collocs,
                bc_collocs=bc_collocs,
                bc_data=bc_data,
                architecture=architecture,
                method=method,
                seed=seed,
                **TRAINING_CONFIG,
                verbose=True
            )
            
            # Store result
            result_entry = {
                'a1': a1,
                'a2': a2,
                'arch_str': str(architecture),
                'method': method,
                'seed': seed,
                'rel_l2_error': result['rel_l2_error'],
                'wall_time': result['wall_time']
            }
            
            results.append(result_entry)
            method_errors.append(result['rel_l2_error'])
        
        # Print summary for this method
        med_error = np.median(method_errors)
        std_error = np.std(method_errors)
        print(f"→ Med L^2: {med_error:.3e} ± {std_error:.3e} \t [{current_run}/{total_runs}]")

print("\n" + "=" * 80)
print(f"Benchmark complete at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total runs: {len(results)}")

## Save Results

In [ ]:
# Convert to DataFrame
df_results = pd.DataFrame(results)

# Save to CSV
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'results/helmholtz_results_{timestamp}.csv'
df_results.to_csv(filename, index=False)
print(f"Results saved to: {filename}")